In [0]:
# === J6 - Data quality tests setup ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

# Import all test modules
import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation',
            'tests.test_ingestion', 'tests.test_quality']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from tests.test_ingestion import TestCleanColumnName, TestIngestCsvToBronzeMemory, TestGenerateSampleDataset
from tests.test_quality import (
    TestRemoveDuplicates, TestHandleNulls, TestFilterInvalidValues,
    TestStandardizeLabels, TestDropConstantColumns, TestAddDerivedColumns,
    TestAddPacketAggregateFeatures, TestAddBinaryTarget, TestAddProtocolName,
    TestEndToEndPipeline,
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, FloatType, TimestampType
)

spark = SparkSession.builder.appName("j6-tests").getOrCreate()

print(f"✓ Spark version: {spark.version}")
print(f"✓ Test modules imported")

In [0]:
# === Custom test runner for Databricks Serverless (UPDATED with SKIPPED support) ===
import os
import inspect
from pathlib import Path

class TestRunner:
    """Test runner that works on Databricks Serverless."""
    
    def __init__(self, spark):
        self.spark = spark
        self.passed = 0
        self.failed = 0
        self.skipped = 0
        self.errors = []
    
    def run_test(self, test_class, test_method_name, skip_if_tmp=False):
        """
        Run a single test method and report result.
        
        Args:
            test_class: the test class (e.g., TestCleanColumnName)
            test_method_name: the unbound method (e.g., TestCleanColumnName.test_xxx)
            skip_if_tmp: if True, skip this test on Databricks Serverless
                         (because it requires /tmp/ file access which is disabled)
        """
        try:
            sig = inspect.signature(test_method_name)
            params = list(sig.parameters.keys())
            
            # Skip tests that require tmp_path on Databricks Serverless
            if skip_if_tmp and 'tmp_path' in params:
                print(f"  ⊘ {test_class.__name__}.{test_method_name.__name__} - SKIPPED (requires /tmp/, not available on Databricks Serverless)")
                self.skipped += 1
                return
            
            instance = test_class.__new__(test_class)
            
            args = {}
            if 'spark' in params:
                args['spark'] = self.spark
            if 'bronze_df' in params:
                args['bronze_df'] = self._create_bronze_df()
            
            method = getattr(instance, test_method_name.__name__)
            method(**args)
            
            print(f"  ✓ {test_class.__name__}.{test_method_name.__name__}")
            self.passed += 1
        except AssertionError as e:
            print(f"  ✗ {test_class.__name__}.{test_method_name.__name__} - {e}")
            self.failed += 1
            self.errors.append((test_class.__name__, test_method_name.__name__, str(e)))
        except Exception as e:
            print(f"  ✗ {test_class.__name__}.{test_method_name.__name__} - ERROR: {e}")
            self.failed += 1
            self.errors.append((test_class.__name__, test_method_name.__name__, f"ERROR: {e}"))
    
    def _create_bronze_df(self):
        """
        Create a RICH Bronze DataFrame with all columns needed for transformation tests.
        Includes packet, flag, IAT, header columns so all 17 Gold features can be created.
        """
        from pyspark.sql.types import (
            StructType, StructField, StringType, IntegerType, 
            LongType, FloatType, TimestampType
        )
        
        schema = StructType([
            StructField("source_ip", StringType(), True),
            StructField("destination_ip", StringType(), True),
            StructField("protocol", IntegerType(), True),
            StructField("port", IntegerType(), True),
            StructField("bytes", LongType(), True),
            StructField("label", StringType(), True),
            StructField("timestamp", TimestampType(), True),
            # Packet columns
            StructField("total_fwd_packets", IntegerType(), True),
            StructField("total_backward_packets", IntegerType(), True),
            StructField("fwd_packets_length_total", FloatType(), True),
            StructField("bwd_packets_length_total", FloatType(), True),
            # Flag columns
            StructField("fin_flag_count", IntegerType(), True),
            StructField("syn_flag_count", IntegerType(), True),
            StructField("rst_flag_count", IntegerType(), True),
            StructField("psh_flag_count", IntegerType(), True),
            StructField("ack_flag_count", IntegerType(), True),
            StructField("cwr_flag_count", IntegerType(), True),
            StructField("ece_flag_count", IntegerType(), True),
            # IAT columns
            StructField("flow_iat_max", FloatType(), True),
            StructField("flow_iat_min", FloatType(), True),
            StructField("active_max", FloatType(), True),
            StructField("active_min", FloatType(), True),
            StructField("idle_max", FloatType(), True),
            StructField("idle_min", FloatType(), True),
            StructField("idle_mean", FloatType(), True),
            StructField("active_mean", FloatType(), True),
            # Header columns
            StructField("fwd_header_length", IntegerType(), True),
            StructField("bwd_header_length", IntegerType(), True),
            StructField("flow_duration", LongType(), True),
            # Metadata
            StructField("_source_file", StringType(), True),
            StructField("_ingestion_ts", TimestampType(), True),
        ])
        
        data = [
            ("192.168.1.1", "8.8.8.8", 6, 80, 1000, "Benign", None,
             10, 5, 500.0, 500.0, 0, 0, 0, 0, 5, 0, 0, 100.0, 10.0, 50.0, 10.0, 0.0, 0.0, 25.0, 25.0,
             100, 100, 1000, "test.csv", None),
            ("10.0.0.5", "8.8.8.8", 17, 53, 500, "DDoS", None,
             100, 0, 5000.0, 0.0, 0, 50, 0, 0, 0, 0, 0, 200.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
             200, 0, 5000, "test.csv", None),
            ("192.168.1.1", "1.1.1.1", 6, 443, 2000, "Benign", None,
             20, 10, 1000.0, 1000.0, 0, 0, 0, 0, 10, 0, 0, 150.0, 20.0, 75.0, 20.0, 0.0, 0.0, 35.0, 35.0,
             150, 150, 2000, "test.csv", None),
            ("172.16.0.1", "8.8.8.8", 1, 0, 100, "PortScan", None,
             5, 5, 50.0, 50.0, 0, 0, 5, 0, 0, 0, 0, 50.0, 10.0, 25.0, 10.0, 0.0, 0.0, 12.5, 12.5,
             50, 50, 100, "test.csv", None),
            ("10.0.0.5", "8.8.8.8", 17, 53, 500, "DDoS", None,
             100, 0, 5000.0, 0.0, 0, 50, 0, 0, 0, 0, 0, 200.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
             200, 0, 5000, "test.csv", None),
        ]
        
        return self.spark.createDataFrame(data, schema)
    
    def summary(self):
        print(f"\n{'=' * 60}")
        print(f"Test Summary: {self.passed} passed, {self.failed} failed, {self.skipped} skipped")
        print(f"{'=' * 60}")
        if self.errors:
            print(f"\nFailed tests:")
            for cls, method, err in self.errors:
                print(f"  - {cls}.{method}: {err}")
        return self.passed, self.failed


# Initialize the runner
runner = TestRunner(spark)
print("✓ Test runner initialized (with SKIPPED support for /tmp/-based tests)")

In [0]:
# === Run all tests (UPDATED to skip tmp_path-based tests on Databricks) ===

print("=" * 60)
print("Running Data Quality Tests")
print("=" * 60)

# === Ingestion tests ===
print("\n--- Tests: TestCleanColumnName ---")
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_strips_leading_whitespace)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_strips_trailing_whitespace)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_converts_internal_spaces_to_underscores)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_converts_to_lowercase)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_handles_slashes)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_handles_dashes)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_handles_already_clean_names)
runner.run_test(TestCleanColumnName, TestCleanColumnName.test_handles_real_cic_ids2017_columns)

print("\n--- Tests: TestIngestCsvToBronzeMemory (skipped on Databricks Serverless) ---")
runner.run_test(TestIngestCsvToBronzeMemory, TestIngestCsvToBronzeMemory.test_returns_dataframe, skip_if_tmp=True)
runner.run_test(TestIngestCsvToBronzeMemory, TestIngestCsvToBronzeMemory.test_adds_metadata_columns, skip_if_tmp=True)
runner.run_test(TestIngestCsvToBronzeMemory, TestIngestCsvToBronzeMemory.test_cleans_column_names, skip_if_tmp=True)

print("\n--- Tests: TestGenerateSampleDataset (skipped on Databricks Serverless) ---")
runner.run_test(TestGenerateSampleDataset, TestGenerateSampleDataset.test_creates_csv_file, skip_if_tmp=True)
runner.run_test(TestGenerateSampleDataset, TestGenerateSampleDataset.test_csv_has_correct_row_count, skip_if_tmp=True)
runner.run_test(TestGenerateSampleDataset, TestGenerateSampleDataset.test_csv_has_expected_columns, skip_if_tmp=True)

# === Cleaning tests ===
print("\n--- Tests: TestRemoveDuplicates ---")
runner.run_test(TestRemoveDuplicates, TestRemoveDuplicates.test_removes_exact_duplicates)
runner.run_test(TestRemoveDuplicates, TestRemoveDuplicates.test_no_duplicates_returns_same_count)

print("\n--- Tests: TestHandleNulls ---")
runner.run_test(TestHandleNulls, TestHandleNulls.test_drops_rows_with_critical_nulls)
runner.run_test(TestHandleNulls, TestHandleNulls.test_fills_non_critical_nulls)

print("\n--- Tests: TestFilterInvalidValues ---")
runner.run_test(TestFilterInvalidValues, TestFilterInvalidValues.test_filters_negative_bytes)
runner.run_test(TestFilterInvalidValues, TestFilterInvalidValues.test_filters_invalid_ports)

print("\n--- Tests: TestStandardizeLabels ---")
runner.run_test(TestStandardizeLabels, TestStandardizeLabels.test_trims_whitespace)
runner.run_test(TestStandardizeLabels, TestStandardizeLabels.test_handles_missing_label_column)

print("\n--- Tests: TestDropConstantColumns ---")
runner.run_test(TestDropConstantColumns, TestDropConstantColumns.test_preserves_label_column)
runner.run_test(TestDropConstantColumns, TestDropConstantColumns.test_drops_only_truly_constant_columns)

print("\n--- Tests: TestAddDerivedColumns ---")
runner.run_test(TestAddDerivedColumns, TestAddDerivedColumns.test_adds_is_attack_column)
runner.run_test(TestAddDerivedColumns, TestAddDerivedColumns.test_adds_attack_category_column)

# === Transformation tests ===
print("\n--- Tests: TestAddPacketAggregateFeatures ---")
runner.run_test(TestAddPacketAggregateFeatures, TestAddPacketAggregateFeatures.test_adds_5_new_features)
runner.run_test(TestAddPacketAggregateFeatures, TestAddPacketAggregateFeatures.test_total_packets_is_sum)
runner.run_test(TestAddPacketAggregateFeatures, TestAddPacketAggregateFeatures.test_handles_division_by_zero)

print("\n--- Tests: TestAddBinaryTarget ---")
runner.run_test(TestAddBinaryTarget, TestAddBinaryTarget.test_benign_becomes_zero)
runner.run_test(TestAddBinaryTarget, TestAddBinaryTarget.test_attacks_become_one)
runner.run_test(TestAddBinaryTarget, TestAddBinaryTarget.test_handles_missing_label)

print("\n--- Tests: TestAddProtocolName ---")
runner.run_test(TestAddProtocolName, TestAddProtocolName.test_maps_tcp)
runner.run_test(TestAddProtocolName, TestAddProtocolName.test_maps_udp)
runner.run_test(TestAddProtocolName, TestAddProtocolName.test_maps_icmp)

# === End-to-end tests ===
print("\n--- Tests: TestEndToEndPipeline ---")
runner.run_test(TestEndToEndPipeline, TestEndToEndPipeline.test_bronze_to_silver_preserves_label)
runner.run_test(TestEndToEndPipeline, TestEndToEndPipeline.test_silver_to_gold_adds_features)
runner.run_test(TestEndToEndPipeline, TestEndToEndPipeline.test_gold_has_is_attack)
runner.run_test(TestEndToEndPipeline, TestEndToEndPipeline.test_gold_has_protocol_name)

# === Summary ===
passed, failed = runner.summary()